In [1]:
# import sys
# sys.path.append(".")

# import os
# import pickle
# from storybot.tokenizer import train_bpe

# vocab_size = 50000

# with open("webbot/owt_train.txt", "rb") as src, open("webbot/owt_bpe_sample.txt", "wb") as dst:
#     dst.write(src.read(1 * 1024**3))

# merge_rules = train_bpe("webbot/owt_bpe_sample.txt", vocab_size, num_processes = 16, num_chunks = 64)

# with open("webbot/merge_rules.pkl", "wb") as f:
#     pickle.dump(merge_rules, f)

In [2]:
import sys
sys.path.append(".")

import os
import pickle
from storybot.tokenizer import train_bpe

vocab_size = 50000
file_path = "webbot/owt_train.txt"
merge_rules = train_bpe(file_path, vocab_size, num_processes = 8, num_chunks = 64)

with open("webbot/merge_rules.pkl", "wb") as f:
    pickle.dump(merge_rules, f)

FileNotFoundError: [Errno 2] No such file or directory: 'webbot/owt_train.txt'

In [ ]:
from storybot.tokenizer import BPETokenizer

tokenizer = BPETokenizer.load_from("webbot/merge_rules.pkl")

tokenizer.encode_file("webbot/owt_train.txt",
                      "webbot/owt_train.bin",
                      num_processes = 8)

tokenizer.encode_file("webbot/owt_valid.txt",
                      "webbot/owt_valid.bin",
                      num_processes = 8)

In [ ]:
import os
import numpy as np
import torch
import torch.nn.functional as F
import torch.distributed as dist # gpu분산 환경 제공
from torch.nn.parallel import DistributedDataParallel as DDP # 모델을 복제해서 동시에 학습시키기 위한 라이브러리
from torch.amp import autocast
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
import wandb # 손실값 등을 대시보드에 등록해서 추후 확인가능하도록 하는 라이브러리
from webbot.model import GPT
from webbot.utils import get_device

In [ ]:
ddp = int(os.environ.get("RANK", -1)) != -1

if ddp:
    dist.init_process_group(backend = "nccl") # nccl = gpu간 통신방법
    local_rank = int(os.environ["LOCAL_RANK"])
    device = torch.device(f"cuda:{local_rank}")
    torch.cuda.set_device(device)
    seed_offset = int(os.environ["RANK"])
else:
    device = get_device()
    local_rank = 0
    seed_offset = 0

is_main = (not ddp) or (local_rank == 0) # 분산환경에서 gpu를 많이 사용하지 않을때 or로 선택해서 가동하게끔
torch.manual_seed(42 + seed_offset)

if is_main:
    wandb.init(project = "webbot-pretrain", config = {})

def get_lr(it, max_lr, warmup_iters, max_iters):
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)

def get_batch(data,
              context_len,
              batch_size,
              device,
              random = True,
              offset = 0):
    if random:
        ix = torch.randint(len(data) - context_len - 1, (batch_size,))
    else:
        ix = torch.arange(offset, offset + batch_size * context_len, context_len)
        ix = ix[ix + context_len + 1 < len(data)]
        if len(ix) == 0:
            return None, None

    x = torch.stack([torch.from_numpy(data[i:i+context_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+context_len+1].astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)

def evaluate(model, val_data, context_len, batch_size, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0.0

    max_start = len(val_data) - context_len - 1
    num_batches = (max_start // context_len) // batch_size + 1

    with torch.no_grad():
        for batch_idx in tqdm(range(num_batches), desc = "Evaluation", leave = False):
            offset = batch_idx * batch_size * context_len
            x, y = get_batch(val_data,
                             context_len,
                             batch_size,
                             device,
                             random = False,
                             offset = offset)
            if x is None:
                break
            with autocast(device_type = "cuda", dtype = torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                       y.view(-1),
                                       reduction = "sum")
            total_loss += loss.item()
            total_tokens += y.numel()
    model.train()
    return total_loss / total_tokens

In [ ]:
vocab_size = 50000
context_len = 1024
embed_dim = 768
n_head = 12
n_kv_head = 4
n_layer = 12 # block의 개수도 12
ff_dim = 2048
theta = 10000

micro_batch_size = 32
accumulation_steps = 4
learning_rate = 6e-4
warmup_iters = 500
max_iters = 100000
grad_clip = 1.0
eval_interval = 1000

if is_main:
    wandb.config.update({"vocab_size": vocab_size,
                         "context_len": context_len,
                         "embed_dim": embed_dim,
                         "n_head": n_head,
                         "n_kv_head": n_kv_head,
                         "n_layer": n_layer,
                         "ff_dim": ff_dim,
                         "theta": theta,
                         "micro_batch_size": micro_batch_size,
                         "accumulation_steps": accumulation_steps,
                         "effective_batch_size": micro_batch_size * accumulation_steps,
                         "learning_rate": learning_rate,
                         "warmup_iters": warmup_iters,
                         "max_iters": max_iters,
                         "grad_clip": grad_clip})

train_data_path = "webbot/owt_train.bin"
valid_data_path = "webbot/owt_valid.bin"
model_save_path = "webbot/model_pretrain.pt"

train_data = np.memmap(train_data_path, dtype = np.uint16, mode = "r")
valid_data = np.memmap(valid_data_path, dtype = np.uint16, mode = "r")

if is_main:
    print(f"학습데이터 : {len(train_data)}")
    print(f"검증데이터 : {len(valid_data)}")

model = GPT(vocab_size,
            context_len,
            embed_dim,
            n_head,
            n_kv_head,
            n_layer,
            ff_dim,
            theta).to(device)

if is_main:
    num_params = sum(p.numel() for p in model.parameters())
    print(f"파라미터수 : {num_params}")

model = torch.compile(model) # 계산과정을 최적화해서 시간을 줄여주는

if ddp:
    model = DDP(model, device_ids = [local_rank])

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

In [ ]:
pbar = tqdm(range(max_iters), disable = not is_main)
val_loss = float("inf")
val_losses = []
val_iters = []

for step in pbar:
    lr = get_lr(step, learning_rate, warmup_iters, max_iters)
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

    optimizer.zero_grad()

    for micro_step in range(accumulation_steps):
        batch_x, batch_y = get_batch(train_data, context_len, micro_batch_size, device)

        with autocast(device_type = "cuda", dtype = torch.bfloat16):
            logits = model(batch_x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   batch_y.view(-1))
            loss = loss / accumulation_steps

        if ddp and micro_step < accumulation_steps - 1:
            with model.no_sync():
                loss.backward()
        else:
            loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    train_loss = loss.item() * accumulation_steps
    if is_main:
        wandb.log({"train/loss": train_loss,
                   "train/lr": lr},
                   step = step)

    if is_main and ((step % eval_interval) == 0 or step == max_iters - 1):
        raw_model = model.module if ddp else model
        val_loss = evaluate(raw_model, valid_data, context_len, micro_batch_size, device)
        val_losses.append(val_loss)
        val_iters.append(step)
        print(f"스텝 {step}: 검증손실 {val_loss:.4f}")
        wandb.log({"val/loss": val_loss}, step = step)
    
    if is_main:
        pbar.set_postfix({"loss": f"{train_loss:.4f}",
                          "val_loss": f"{val_loss:.4f}",
                          "lr": f"{lr:.2e}"})

if is_main:
    raw_model = model.module if ddp else model
    raw_model.save(model_save_path)
    print("모델 저장")

if is_main:
    wandb.finish()

if ddp:
    dist.destroy_process_group()

In [ ]:
from webbot.model import GPT
from storybot.tokenizer import BPETokenizer
from webbot.utils import generate, get_device

device = get_device()
model_path = "webbot/model_pretrain.pt"
tokenizer_path = "webbot/merge_rules.pkl"
max_new_tokens = 100

temperature = 0.5

prompts = ["In 1991, Linus Torvalds created",
           "Monday, Tuesday, Wednesday,",
           "Python was created by",
           "Machine learning is defined as",
           "The capital of Korea is"]

tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT.load_from(model_path, device = device)

for i, prompt in enumerate(prompts, 1):
    print(f"프롬프트 : {prompt}")
    response = generate(model, tokenizer, prompt, max_new_tokens, temperature)
    print(response)